# Notebook 01: PDF Extraction - Bronze Layer

## Overview
This notebook extracts data from cholera situation report PDFs and loads them into Bronze layer Delta tables.

## Prerequisites
- PDFs uploaded to `/lakehouse/default/Files/bronze/pdfs/{year}/wk{week}/` (Fabric) or `data/sample/` (Local)
- Lakehouse attached to notebook (Fabric only)

## Inputs
- PDF files: `cholera_sitrep_{year}_wk{week}.pdf`
- Reference data: `country_codes_iso3166.csv`

## Outputs
- Delta table: `bronze.report_summary` (report-level data)
- Delta table: `bronze.country_weekly` (country-level weekly data)
- Delta table: `bronze.extraction_metadata` (extraction logs)

## Execution Time
~2-3 minutes for 3 PDFs

In [1]:
# ============================================
# ENVIRONMENT DETECTION & CONFIGURATION
# ============================================

import os
import sys
from pathlib import Path

# Auto-detect environment
IS_FABRIC = os.path.exists('/lakehouse/default')

if IS_FABRIC:
    print("🌐 Running in Microsoft Fabric")
    BRONZE_PDF_PATH = "/lakehouse/default/Files/bronze/pdfs"
    REFERENCE_PATH = "/lakehouse/default/Files/reference"
    BRONZE_TABLE_PATH = "/lakehouse/default/Tables/bronze"
else:
    print("💻 Running locally")
    # Add src to path for local imports
    project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    sys.path.insert(0, str(project_root / 'src'))
    
    BRONZE_PDF_PATH = str(project_root / "data" / "sample")
    REFERENCE_PATH = str(project_root / "data" / "reference")
    BRONZE_TABLE_PATH = str(project_root / "data" / "bronze_tables")
    
    # Create output directory
    Path(BRONZE_TABLE_PATH).mkdir(parents=True, exist_ok=True)

print(f"PDF Path: {BRONZE_PDF_PATH}")
print(f"Reference Path: {REFERENCE_PATH}")
print(f"Output Path: {BRONZE_TABLE_PATH}")

💻 Running locally
PDF Path: D:\Projects\cholera-cdr-mvp\data\sample
Reference Path: D:\Projects\cholera-cdr-mvp\data\reference
Output Path: D:\Projects\cholera-cdr-mvp\data\bronze_tables


In [2]:
# ============================================
# IMPORTS
# ============================================

import pdfplumber
import pandas as pd
import re
from datetime import datetime
from typing import Dict, List, Any, Optional
import json
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ Imports successful")

✅ Imports successful


In [3]:
# ============================================
# PDF EXTRACTION FUNCTIONS
# ============================================

def extract_kpi_panel(pdf_path: str) -> Dict[str, Any]:
    """
    Extract KPI panel data from cholera situation report PDF.
    
    Args:
        pdf_path: Path to PDF file
        
    Returns:
        Dictionary with extracted KPI data
    """
    kpi_data = {
        'confirmed_cases': None,
        'suspected_cases': None,
        'deaths': None,
        'cfr_percent': None,
        'affected_countries': None
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Extract text from first page (KPI panel location)
            first_page = pdf.pages[0]
            text = first_page.extract_text()
            
            if not text:
                logger.warning(f"No text extracted from {pdf_path}")
                return kpi_data
            
            # Extract confirmed cases - look for number before "Suspected Cases"
            # Pattern: "Confirmed Cases Suspected Cases Deaths CFR (%)\n1075 1505 26 2.42"
            lines = text.split('\n')
            for i, line in enumerate(lines):
                # Look for the header line
                if 'Confirmed Cases' in line and 'Suspected Cases' in line:
                    # Next line should have the numbers
                    if i + 1 < len(lines):
                        numbers_line = lines[i + 1].strip()
                        # Split by whitespace and extract numbers
                        numbers = numbers_line.split()
                        if len(numbers) >= 4:
                            try:
                                kpi_data['confirmed_cases'] = int(numbers[0].replace(',', ''))
                                kpi_data['suspected_cases'] = int(numbers[1].replace(',', ''))
                                kpi_data['deaths'] = int(numbers[2].replace(',', ''))
                                kpi_data['cfr_percent'] = float(numbers[3].replace(',', ''))
                            except (ValueError, IndexError) as e:
                                logger.warning(f"Error parsing KPI numbers: {e}")
            
            # Extract affected countries - look for "Affected Countries" line
            for i, line in enumerate(lines):
                if 'Affected Countries' in line:
                    # Next line should have the number
                    if i + 1 < len(lines):
                        numbers_line = lines[i + 1].strip()
                        numbers = numbers_line.split()
                        if len(numbers) >= 1:
                            try:
                                kpi_data['affected_countries'] = int(numbers[0])
                            except ValueError:
                                pass
                    break
                    
    except Exception as e:
        logger.error(f"Error extracting KPI from {pdf_path}: {e}")
        raise
    
    return kpi_data


def extract_country_breakdown(pdf_path: str) -> List[Dict[str, Any]]:
    """
    Extract country-level breakdown from PDF text.
    
    Args:
        pdf_path: Path to PDF file
        
    Returns:
        List of dictionaries with country-level data
    """
    country_data = []
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Extract all text
            full_text = ""
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    full_text += page_text + "\n"
            
            if not full_text:
                return []
            
            # Look for "Country-Level Summary:" section
            if "Country-Level Summary:" in full_text:
                # Split by country entries
                # Pattern: "Zimbabwe: Reported 355 cumulative cases with 7 deaths."
                country_pattern = r'([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*?):\s+(?:Reported\s+)?(\d+)\s+(?:cumulative\s+)?cases?\s+(?:with\s+)?(?:and\s+)?(\d+)\s+deaths?'
                
                matches = re.finditer(country_pattern, full_text)
                
                for match in matches:
                    country_name = match.group(1).strip()
                    cases = int(match.group(2))
                    deaths = int(match.group(3))
                    
                    country_record = {
                        'country_name': country_name,
                        'confirmed_cases': cases,
                        'suspected_cases': None,  # Not always available
                        'deaths': deaths
                    }
                    
                    country_data.append(country_record)
            
            # Also try table extraction as fallback
            if len(country_data) == 0:
                for page in pdf.pages:
                    tables = page.extract_tables()
                    
                    for table in tables:
                        if not table or len(table) < 2:
                            continue
                        
                        # Check if this is a country breakdown table
                        header = [str(cell).lower() if cell else '' for cell in table[0]]
                        
                        if 'country' in ' '.join(header):
                            # Process country data rows
                            for row in table[1:]:
                                if not row or not row[0]:
                                    continue
                                
                                country_name = str(row[0]).strip()
                                
                                # Skip header rows or totals
                                if country_name.lower() in ['country', 'total', 'grand total']:
                                    continue
                                
                                # Extract numeric values
                                country_record = {
                                    'country_name': country_name,
                                    'confirmed_cases': extract_number(row[1] if len(row) > 1 else None),
                                    'suspected_cases': extract_number(row[2] if len(row) > 2 else None),
                                    'deaths': extract_number(row[3] if len(row) > 3 else None)
                                }
                                
                                country_data.append(country_record)
                            
    except Exception as e:
        logger.warning(f"Error extracting country breakdown from {pdf_path}: {e}")
        # Return empty list rather than failing
        return []
    
    return country_data


def extract_number(value: Any) -> Optional[int]:
    """
    Extract numeric value from string, handling commas and None.
    """
    if value is None or str(value).strip() == '':
        return None
    
    try:
        # Remove commas and convert to int
        clean_value = str(value).replace(',', '').strip()
        return int(float(clean_value))
    except (ValueError, AttributeError):
        return None


def extract_metadata(pdf_path: str) -> Dict[str, Any]:
    """
    Extract metadata from PDF file.
    
    Args:
        pdf_path: Path to PDF file
        
    Returns:
        Dictionary with PDF metadata
    """
    metadata = {
        'file_name': Path(pdf_path).name,
        'file_size_bytes': None,
        'page_count': None,
        'extraction_timestamp': datetime.now().isoformat()
    }
    
    try:
        # Get file size
        metadata['file_size_bytes'] = Path(pdf_path).stat().st_size
        
        # Get page count
        with pdfplumber.open(pdf_path) as pdf:
            metadata['page_count'] = len(pdf.pages)
            
    except Exception as e:
        logger.warning(f"Error extracting metadata from {pdf_path}: {e}")
    
    return metadata


print("✅ Extraction functions defined")

✅ Extraction functions defined


In [4]:
# ============================================
# DISCOVER PDF FILES
# ============================================

def find_pdf_files(base_path: str) -> List[str]:
    """
    Find all PDF files in the bronze directory.
    
    Args:
        base_path: Base path to search for PDFs
        
    Returns:
        List of PDF file paths
    """
    pdf_files = []
    base_path_obj = Path(base_path)
    
    if base_path_obj.exists():
        # Recursively find all PDF files
        pdf_files = list(base_path_obj.rglob('*.pdf'))
        pdf_files = [str(f) for f in pdf_files]
    else:
        logger.warning(f"Path does not exist: {base_path}")
    
    return pdf_files


# Find all PDFs
pdf_files = find_pdf_files(BRONZE_PDF_PATH)

print(f"\n📄 Found {len(pdf_files)} PDF files:")
for pdf in pdf_files:
    print(f"  - {Path(pdf).name}")

if len(pdf_files) == 0:
    print("\n⚠️  No PDF files found. Please check the path and upload PDFs.")


📄 Found 3 PDF files:
  - cholera_sitrep_2025_wk06.pdf
  - cholera_sitrep_2025_wk07.pdf
  - cholera_sitrep_2025_wk08.pdf


In [5]:
# ============================================
# EXTRACT DATA FROM ALL PDFs
# ============================================

report_summaries = []
country_breakdowns = []
extraction_logs = []

print("\n🔄 Starting PDF extraction...\n")

for pdf_path in pdf_files:
    try:
        logger.info(f"Processing: {Path(pdf_path).name}")
        
        # Extract report ID from filename (e.g., cholera_sitrep_2025_wk06.pdf -> 2025_wk06)
        filename = Path(pdf_path).stem
        match = re.search(r'(\d{4})_wk(\d{2})', filename)
        
        if match:
            year = int(match.group(1))
            week = int(match.group(2))
            report_id = f"{year}_wk{week:02d}"
        else:
            logger.warning(f"Could not parse report ID from filename: {filename}")
            report_id = filename
            year = None
            week = None
        
        # Extract KPI data
        kpi_data = extract_kpi_panel(pdf_path)
        
        # Extract country breakdown
        countries = extract_country_breakdown(pdf_path)
        
        # Extract metadata
        metadata = extract_metadata(pdf_path)
        
        # Create report summary record
        report_summary = {
            'report_id': report_id,
            'epi_year': year,
            'epi_week': week,
            'report_date': None,  # Will be derived from epi week in Silver layer
            'confirmed_cases': kpi_data.get('confirmed_cases'),
            'suspected_cases': kpi_data.get('suspected_cases'),
            'deaths': kpi_data.get('deaths'),
            'cfr_percent': kpi_data.get('cfr_percent'),
            'affected_countries': kpi_data.get('affected_countries'),
            'source_file': Path(pdf_path).name,
            'extraction_timestamp': datetime.now().isoformat(),
            'country_breakdown_count': len(countries)
        }
        
        report_summaries.append(report_summary)
        
        # Add report_id to country records
        for country in countries:
            country['report_id'] = report_id
            country['epi_year'] = year
            country['epi_week'] = week
            country['extraction_timestamp'] = datetime.now().isoformat()
        
        country_breakdowns.extend(countries)
        
        # Log extraction
        extraction_log = {
            'report_id': report_id,
            'file_name': metadata['file_name'],
            'file_size_bytes': metadata['file_size_bytes'],
            'page_count': metadata['page_count'],
            'extraction_timestamp': metadata['extraction_timestamp'],
            'kpi_extracted': kpi_data.get('confirmed_cases') is not None,
            'countries_extracted': len(countries),
            'status': 'SUCCESS'
        }
        
        extraction_logs.append(extraction_log)
        
        print(f"✅ {report_id}: {kpi_data.get('confirmed_cases', 0):,} cases, {len(countries)} countries")
        
    except Exception as e:
        logger.error(f"Failed to process {Path(pdf_path).name}: {e}")
        
        # Log failure
        extraction_logs.append({
            'report_id': Path(pdf_path).stem,
            'file_name': Path(pdf_path).name,
            'extraction_timestamp': datetime.now().isoformat(),
            'status': 'FAILED',
            'error_message': str(e)
        })

print(f"\n✅ Extraction complete: {len(report_summaries)} reports, {len(country_breakdowns)} country records")

2026-02-11 05:26:39,966 - INFO - Processing: cholera_sitrep_2025_wk06.pdf



🔄 Starting PDF extraction...



2026-02-11 05:26:40,112 - INFO - Processing: cholera_sitrep_2025_wk07.pdf


✅ 2025_wk06: 1,075 cases, 2 countries


2026-02-11 05:26:40,286 - INFO - Processing: cholera_sitrep_2025_wk08.pdf


✅ 2025_wk07: 1,379 cases, 2 countries
✅ 2025_wk08: 1,993 cases, 2 countries

✅ Extraction complete: 3 reports, 6 country records


In [6]:
# ============================================
# CONVERT TO DATAFRAMES
# ============================================

df_report_summary = pd.DataFrame(report_summaries)
df_country_weekly = pd.DataFrame(country_breakdowns)
df_extraction_metadata = pd.DataFrame(extraction_logs)

print("\n📊 DataFrames created:")
print(f"  - report_summary: {len(df_report_summary)} rows")
print(f"  - country_weekly: {len(df_country_weekly)} rows")
print(f"  - extraction_metadata: {len(df_extraction_metadata)} rows")

# Display sample data
print("\n📋 Sample Report Summary:")
display(df_report_summary.head())

if len(df_country_weekly) > 0:
    print("\n📋 Sample Country Weekly:")
    display(df_country_weekly.head())


📊 DataFrames created:
  - report_summary: 3 rows
  - country_weekly: 6 rows
  - extraction_metadata: 3 rows

📋 Sample Report Summary:


,report_id,epi_year,epi_week,report_date,confirmed_cases,suspected_cases,deaths,cfr_percent,affected_countries,source_file,extraction_timestamp,country_breakdown_count
0,2025_wk06,2025,6,None,1075,1505,26,2.42,5,cholera_sitrep_2025_wk06.pdf,2026-02-11T05:26:40.110828,2
1,2025_wk07,2025,7,None,1379,1930,30,2.18,9,cholera_sitrep_2025_wk07.pdf,2026-02-11T05:26:40.285546,2
2,2025_wk08,2025,8,None,1993,2790,35,1.76,6,cholera_sitrep_2025_wk08.pdf,2026-02-11T05:26:40.452397,2



📋 Sample Country Weekly:


,country_name,confirmed_cases,suspected_cases,deaths,report_id,epi_year,epi_week,extraction_timestamp
0,Zimbabwe,355,None,7,2025_wk06,2025,6,2026-02-11T05:26:40.110828
1,Zambia,245,None,9,2025_wk06,2025,6,2026-02-11T05:26:40.110828
2,Zimbabwe,409,None,8,2025_wk07,2025,7,2026-02-11T05:26:40.285546
3,Zambia,372,None,5,2025_wk07,2025,7,2026-02-11T05:26:40.285546
4,Zimbabwe,429,None,11,2025_wk08,2025,8,2026-02-11T05:26:40.452397


In [7]:
# ============================================
# SAVE TO DELTA TABLES (BRONZE LAYER)
# ============================================

print("\n💾 Saving to Bronze layer...\n")

try:
    if IS_FABRIC:
        # Fabric: Use PySpark to write Delta tables
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        
        # Convert pandas to Spark DataFrames
        spark_report_summary = spark.createDataFrame(df_report_summary)
        spark_country_weekly = spark.createDataFrame(df_country_weekly)
        spark_extraction_metadata = spark.createDataFrame(df_extraction_metadata)
        
        # Write to Delta tables (overwrite mode for MVP)
        spark_report_summary.write.format("delta").mode("overwrite").saveAsTable("bronze.report_summary")
        spark_country_weekly.write.format("delta").mode("overwrite").saveAsTable("bronze.country_weekly")
        spark_extraction_metadata.write.format("delta").mode("overwrite").saveAsTable("bronze.extraction_metadata")
        
        print("✅ Delta tables created in Fabric Lakehouse")
        
    else:
        # Local: Save as Parquet (Delta-compatible format)
        # Convert datetime columns to strings to avoid pyarrow compatibility issues
        df_report_summary_save = df_report_summary.copy()
        df_country_weekly_save = df_country_weekly.copy()
        df_extraction_metadata_save = df_extraction_metadata.copy()
        
        # Convert timestamp columns to strings
        timestamp_cols_reports = ['extraction_timestamp']
        for col in timestamp_cols_reports:
            if col in df_report_summary_save.columns:
                df_report_summary_save[col] = df_report_summary_save[col].astype(str)
        
        timestamp_cols_countries = ['extraction_timestamp']
        for col in timestamp_cols_countries:
            if col in df_country_weekly_save.columns:
                df_country_weekly_save[col] = df_country_weekly_save[col].astype(str)
        
        timestamp_cols_metadata = ['extraction_timestamp']
        for col in timestamp_cols_metadata:
            if col in df_extraction_metadata_save.columns:
                df_extraction_metadata_save[col] = df_extraction_metadata_save[col].astype(str)
        
        # Save to Parquet
        df_report_summary_save.to_parquet(
            Path(BRONZE_TABLE_PATH) / "report_summary.parquet",
            index=False,
            engine='pyarrow'
        )
        
        df_country_weekly_save.to_parquet(
            Path(BRONZE_TABLE_PATH) / "country_weekly.parquet",
            index=False,
            engine='pyarrow'
        )
        
        df_extraction_metadata_save.to_parquet(
            Path(BRONZE_TABLE_PATH) / "extraction_metadata.parquet",
            index=False,
            engine='pyarrow'
        )
        
        print(f"✅ Parquet files saved to: {BRONZE_TABLE_PATH}")
        print(f"   - report_summary.parquet ({len(df_report_summary_save)} rows)")
        print(f"   - country_weekly.parquet ({len(df_country_weekly_save)} rows)")
        print(f"   - extraction_metadata.parquet ({len(df_extraction_metadata_save)} rows)")
        
except Exception as e:
    logger.error(f"Error saving to Bronze layer: {e}")
    raise

print("\n✅ Bronze layer extraction complete!")


💾 Saving to Bronze layer...

✅ Parquet files saved to: D:\Projects\cholera-cdr-mvp\data\bronze_tables
   - report_summary.parquet (3 rows)
   - country_weekly.parquet (6 rows)
   - extraction_metadata.parquet (3 rows)

✅ Bronze layer extraction complete!


## Validation & Testing

In [8]:
# ============================================
# VALIDATION & TESTING
# ============================================

print("\n🔍 Running validation checks...\n")

# Test 1: Row count validation
expected_reports = len(pdf_files)
actual_reports = len(df_report_summary)
assert actual_reports == expected_reports, f"Expected {expected_reports} reports, got {actual_reports}"
print(f"✅ Report count: {actual_reports} (expected: {expected_reports})")

# Test 2: Required fields present
required_fields = ['report_id', 'confirmed_cases', 'deaths', 'source_file']
for field in required_fields:
    assert field in df_report_summary.columns, f"Missing required field: {field}"
print(f"✅ All required fields present: {required_fields}")

# Test 3: No null report IDs
null_report_ids = df_report_summary['report_id'].isnull().sum()
assert null_report_ids == 0, f"Found {null_report_ids} null report IDs"
print(f"✅ No null report IDs")

# Test 4: Extraction success rate
successful_extractions = df_extraction_metadata[df_extraction_metadata['status'] == 'SUCCESS'].shape[0]
success_rate = (successful_extractions / len(df_extraction_metadata)) * 100 if len(df_extraction_metadata) > 0 else 0
print(f"✅ Extraction success rate: {success_rate:.1f}% ({successful_extractions}/{len(df_extraction_metadata)})")

# Test 5: Data quality summary
print("\n📊 Data Quality Summary:")
print(f"  - Reports with confirmed cases: {df_report_summary['confirmed_cases'].notna().sum()}")
print(f"  - Reports with deaths: {df_report_summary['deaths'].notna().sum()}")
print(f"  - Reports with CFR: {df_report_summary['cfr_percent'].notna().sum()}")
print(f"  - Total country records: {len(df_country_weekly)}")

print("\n✅ All validation checks passed!")


🔍 Running validation checks...

✅ Report count: 3 (expected: 3)
✅ All required fields present: ['report_id', 'confirmed_cases', 'deaths', 'source_file']
✅ No null report IDs
✅ Extraction success rate: 100.0% (3/3)

📊 Data Quality Summary:
  - Reports with confirmed cases: 3
  - Reports with deaths: 3
  - Reports with CFR: 3
  - Total country records: 6

✅ All validation checks passed!


## Next Steps

1. **Review extracted data** in the DataFrames above
2. **Check extraction logs** for any failures or warnings
3. **Proceed to Notebook 02** for Silver layer transformation

## Outputs Created

- `bronze.report_summary` - Report-level summary data
- `bronze.country_weekly` - Country-level weekly data
- `bronze.extraction_metadata` - Extraction logs and metadata